<a href="https://colab.research.google.com/github/DeviShree2006/ITA0620/blob/main/Candidate_Elimination_Algorithm_using_play_tennis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Candidate-Elimination Algorithm
# Dataset: Play Tennis
# Attributes: Outlook, Temperature, Humidity, Wind
# + = Play Tennis (Yes)      - = Don't play (No)

data = [
    ["Sunny",    "Hot",  "High",   "Weak",   "-"],
    ["Sunny",    "Hot",  "High",   "Strong", "-"],
    ["Overcast", "Hot",  "High",   "Weak",   "+"],
    ["Rain",     "Mild", "High",   "Weak",   "+"],
    ["Rain",     "Cool", "Normal", "Weak",   "+"],
    ["Rain",     "Cool", "Normal", "Strong", "-"],
    ["Overcast", "Cool", "Normal", "Strong", "+"],
    ["Sunny",    "Mild", "High",   "Weak",   "-"],
    ["Sunny",    "Cool", "Normal", "Weak",   "+"],
    ["Rain",     "Mild", "Normal", "Weak",   "+"],
    ["Sunny",    "Mild", "Normal", "Strong", "+"],
    ["Overcast", "Mild", "High",   "Strong", "+"],
    ["Overcast", "Hot",  "Normal", "Weak",   "+"],
    ["Rain",     "Mild", "High",   "Strong", "-"],
]

# True  = all 14 rows (not learnable by one conjunctive rule -> version space collapses)
# False = 7-row subset (Overcast = Yes, Sunny = No) -> algorithm converges
USE_ALL_ROWS = False
if not USE_ALL_ROWS:
    data = [data[i] for i in (0, 1, 2, 6, 7, 11, 12)]

attributes = ["Outlook", "Temperature", "Humidity", "Wind"]
n = len(attributes)

# All possible values of each attribute (needed to specialize G)
domains = [sorted({row[i] for row in data}) for i in range(n)]

S = ["Ø"] * n          # most specific hypothesis
G = [["?"] * n]        # most general hypothesis


def covers(hypothesis, example):
    """True if the hypothesis classifies the example as positive."""
    for h, x in zip(hypothesis, example):
        if h == "Ø" or (h != "?" and h != x):
            return False
    return True


def more_general_or_equal(h1, h2):
    """True if h1 is more general than or equal to h2."""
    for a, b in zip(h1, h2):
        if a == "?" or b == "Ø":     # "?" matches anything, everything matches Ø
            continue
        if a != b:
            return False
    return True


def specialize_g(g, example, S):
    """Minimal specializations of g that exclude the negative example
    but stay more general than S."""
    result = []
    for i in range(n):
        if g[i] == "?":
            for value in domains[i]:
                if value != example[i]:
                    h = g.copy()
                    h[i] = value
                    if more_general_or_equal(h, S) and h not in result:
                        result.append(h)
    return result


print("INITIAL")
print("S =", S)
print("G =", G)
print("-" * 60)

for index, row in enumerate(data):
    example, target = row[:-1], row[-1]
    print(f"\nProcessing D{index + 1}: {example} -> {target}")

    if target == "+":                                   # positive example
        G = [g for g in G if covers(g, example)]        # drop inconsistent G
        if S == ["Ø"] * n:
            S = example.copy()
        else:
            for i in range(n):
                if S[i] != example[i]:
                    S[i] = "?"                          # generalize S
    else:                                               # negative example
        if covers(S, example):
            print("S covers the negative example -> version space collapses.")
        new_G = []
        for g in G:
            if covers(g, example):
                new_G.extend(specialize_g(g, example, S))
            else:
                new_G.append(g)
        G = []
        for g in new_G:                                 # remove duplicates
            if g not in G:
                G.append(g)
        # keep only the maximally general members
        G = [g for g in G
             if not any(o != g and more_general_or_equal(o, g) for o in G)]

    print("S =", S)
    print("G =", G)

print("\n" + "=" * 60)
print("FINAL VERSION SPACE BOUNDARIES")
print("=" * 60)
print("\nSpecific Boundary (S):")
print(S)
print("\nGeneral Boundary (G):")
if G:
    for g in G:
        print(g)
else:
    print("(empty)  -> no single conjunctive hypothesis fits all 14 examples")

INITIAL
S = ['Ø', 'Ø', 'Ø', 'Ø']
G = [['?', '?', '?', '?']]
------------------------------------------------------------

Processing D1: ['Sunny', 'Hot', 'High', 'Weak'] -> -
S = ['Ø', 'Ø', 'Ø', 'Ø']
G = [['Overcast', '?', '?', '?'], ['?', 'Cool', '?', '?'], ['?', 'Mild', '?', '?'], ['?', '?', 'Normal', '?'], ['?', '?', '?', 'Strong']]

Processing D2: ['Sunny', 'Hot', 'High', 'Strong'] -> -
S = ['Ø', 'Ø', 'Ø', 'Ø']
G = [['Overcast', '?', '?', '?'], ['?', 'Cool', '?', '?'], ['?', 'Mild', '?', '?'], ['?', '?', 'Normal', '?']]

Processing D3: ['Overcast', 'Hot', 'High', 'Weak'] -> +
S = ['Overcast', 'Hot', 'High', 'Weak']
G = [['Overcast', '?', '?', '?']]

Processing D4: ['Overcast', 'Cool', 'Normal', 'Strong'] -> +
S = ['Overcast', '?', '?', '?']
G = [['Overcast', '?', '?', '?']]

Processing D5: ['Sunny', 'Mild', 'High', 'Weak'] -> -
S = ['Overcast', '?', '?', '?']
G = [['Overcast', '?', '?', '?']]

Processing D6: ['Overcast', 'Mild', 'High', 'Strong'] -> +
S = ['Overcast', '?', '?', '?'